In [ ]:
# Project Scenario: Smart Waste Classification System
# A city municipality wants to build an AI-powered waste segregation system that can automatically classify images of waste into:
# - Recyclable Waste
# - Organic Waste
# - Non-Recyclable Waste

# Dataset: https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification

# Folder structure:
# dataset/
#    train/
#        recyclable/
#        organic/
#        non_recyclable/
#    validation/
#        recyclable/
#        organic/
#        non_recyclable/

In [ ]:
# Task 1: Dataset Structure
import os

dataset_path = "dataset"

for split in ["train", "validation"]:
    for category in ["Recyclable", "Organic", "Non-Recyclable"]:
        path = os.path.join(dataset_path, split, category)
        if os.path.exists(path):
            count = len(os.listdir(path))
            print(f"{split}/{category}: {count} images")
        else:
            print(f"{split}/{category}: folder not found")

In [ ]:
# Task 2: Data Preprocessing
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    'dataset/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    'dataset/validation',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

print("Classes:", train_generator.class_indices)

In [ ]:
# Task 3: CNN Model Development
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Task 4: Model Evaluation
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# After training:
# y_pred = model.predict(val_generator)
# y_pred_classes = np.argmax(y_pred, axis=1)
# y_true = val_generator.classes
# cm = confusion_matrix(y_true, y_pred_classes)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm)
# disp.plot()
# plt.show()

print("Evaluate model after training")

In [ ]:
# Task 5: Transfer Learning with MobileNetV2
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze base layers

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(3, activation='softmax')(x)

transfer_model = Model(inputs=base_model.input, outputs=output)
transfer_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
transfer_model.summary()